# Image Analysis Pipeline

Once you have obtained an accurate enough segmentation of nuclei in your volumetric dataset, extract the features of interest and analyse these measurements in the regions of interest (ROI) that you select. The pipeline goes as follows:
1. Extract features
2. Select ROI
3. Measure packing fraction in ROI

### Import packages

In [ ]:
#from __future__ import print_function, unicode_literals, absolute_import, division
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tifffile import imread, imsave, imwrite
from skimage.measure import regionprops
from PIL import Image
from PIL.TiffTags import TAGS
from datetime import date, datetime
from utils.preprocess import get_spatial_calib_tiff
from utils.features import run
from utils.napariviewer import select_polygon, zetaslice, eliminate_unwanted_objects

date = str(date.today())#[:10]
print(date)

## 1. Extract features

The features that you can extract are the following:
+ single nuclear features: size, axes, axis of elongation (nematics)
+ neighbour statistics: number of neighbors for different definitions
+ local averaging: color maps of your dataset with average values for different measurements

In [ ]:
### Set up the type of features you want to extract
Extract_singleNucleusFeatures = True,
Perform_NeighbourStatistics = True,
Perform_LocalAveraging = False,

### Define input directory where StarDist-3D predictions are
inputdir = "D:/Lucrezia/to_analyse/hdac" # "path/to/predictions"

### Create the directories where all measurements are stored after feature extraction 
outputdir = os.path.join(inputdir, 'ANALYSIS')
os.makedirs(outputdir, exist_ok=True)

isotropicDir = outputdir+'/isotropic_masks'
os.makedirs(isotropicDir, exist_ok=True)

dataDir = outputdir+'/measurements/nucleardata'
os.makedirs(dataDir, exist_ok=True)

nematicDir = outputdir+'/measurements/nematics'
os.makedirs(nematicDir, exist_ok=True)

neighDir = outputdir+'/measurements/neighbours'
os.makedirs(neighDir, exist_ok=True)



In [ ]:

### Iterate through all segmented predictions and extract the selected features of interest
files = sorted(glob(os.path.join(inputdir,'*.tif')))
for i in range(0, len(files)):
    filename = os.path.basename(files[i])
    print("Processing file {} of {} - filename: {}".format(i+1, len(files), filename))

    run(inputdir, outputdir, files[i], dilations = [0, 1, 2, 4],
        dataDir = dataDir, nemDir = nematicDir, neighDir=neighDir,
        singleNucleusFeatures = Extract_singleNucleusFeatures, #True,
        NeighbourStatistics = Perform_NeighbourStatistics, #False,
        LocalAveraging = Perform_LocalAveraging, #True,
        isotropicdata = False,
        )


## 2. Select the regions of interest (ROIs)
In the next cell, a napari GUI will open so that you can modify a pre-selected rectangle to a polygon that will cover the region of interest. All nuclei either touching or being included within the polygon will be saved as part of the ROI. You will then need to close the Napari GUI and input the lower and upper limits for the z axis. Another Napari GUI will open to allw you to select any nucleus that has been errouneously included in the ROI. To do this, you will have to add a point on the nucleus you want to eliminate. Once you are done, you can close the GUI so that the new image can be saved. 

In [ ]:

### Define the directory where isotropic files are
filepath = isotropicDir
rois_dir = os.path.join(outputdir, 'rois')
os.makedirs(rois_dir, exist_ok=True)

### Iterate through the list of images in the input directory
files = sorted(glob(filepath+'/*.tif'))
print("Found {} files in input directory".format(len(files)))

for i in range(len(files)):
    ### Get voxel resolution
    resolution = get_spatial_calib_tiff(files[i])

    ### Read image
    img = imread(files[i])
    
    ### Select region of interest
    new_img = select_polygon(img)
    plt.imshow(np.max(new_img, axis=0))
    plt.show()
   
    ### Slice the region of interest in z
    roi = zetaslice(new_img)

    ### Select nuclei that should not be part of the roi
    f_roi = eliminate_unwanted_objects(roi)
    f_roi = np.array(f_roi, dtype='uint16')

    ### Save files
    filename = os.path.basename(files[i])
    dstPathname = os.path.join(rois_dir, filename)
    imwrite(dstPathname, f_roi, imagej=True, resolution=(1/resolution[2], 1/resolution[1]), metadata={'spacing': resolution[0],'axes': 'ZYX','unit': 'um'}) #image for imagej is like xyz no?

    ids = pd.Series(np.unique(f_roi))
    ids.to_csv(dstPathname[:-4]+'.csv')


## 3. Quantify nuclear packing in ROIs

As a first step, select the nuclei that belong to the ROI. This is necessary for the next steps.

In [ ]:
# isotropicDir = 'path/to/isotropic/images'
# rois_dir = 'path/to/rois/
isoimages_files = sorted(glob(os.path.join(isotropicDir, '*.tif')))
isoimages = list(map(imread, isoimages_files))
resolutions = list(map(get_spatial_calib_tiff, isoimages_files))

labels_files = sorted(glob(os.path.join(rois_dir, '*.csv')))
labels = list(map(pd.read_csv, labels_files))

assert all(os.path.basename(x)[:-4]==os.path.basename(y)[:-4] for x,y in zip(isoimages_files, labels_files))

roi_images = []
for i in range(len(isoimages_files)):
    img = np.asarray(isoimages[i], dtype=int)
    new_img = np.zeros_like(img)
    props = regionprops(img)
    
    if 'label' not in labels[i].columns:
        labels[i].rename(columns = {'0':'label'}, inplace = True)
    els = list(labels[i]['label'])
    
    for i in range(len(props)):
        if props[i].label in els:
            bbox = props[i].bbox
            z_min, y_min, x_min, z_max, y_max, x_max = bbox
            roi = img[z_min:z_max, y_min:y_max, x_min:x_max]
            mask = (roi == props[i].label)
            new_img[z_min:z_max,y_min:y_max, x_min:x_max][mask] = roi[mask]
    roi_images.append(new_img)
    

print(len(roi_images))
plt.imshow(np.max(roi_images[0], axis=0))
plt.show()

#### 3.1. Measure nuclear packing fractions and nuclear concentrations

For each ROI, a mask is produced to cover the inner volume of the ROI and compute both the nuclear volume fraction and the nuclear concentration. 

In [ ]:
from nuclear_packing import measure_volume_fraction_from_masking

rois_files = sorted(glob(os.path.join(inputdir, '*.tif')))
roi_images = list(map(imread, rois_files))
resolutions = list(map(get_spatial_calib_tiff, rois_files))
rois_dir = os.path.join(outputdir, 'rois')
### create directory to collect the packing fraction measurements
packing_dir = os.path.join(rois_dir, 'packing')
os.makedirs(packing_dir, exist_ok=True)

### initialize dictionary
myDict = dict({'hpf': [],  'volume_fraction':[],'density':[], "filename":[]})
name_csv = os.path.join(packing_dir, date+"_volume_fraction.csv")

### iterate through each roi image
for i in range( len(rois_files)):
    filename = os.path.basename(rois_files[i])
    stage = int(filename[filename.find("hpf")-2:filename.find("hpf")])

    # voxel sizes
    voxel_size = resolutions[i]
    roi = roi_images[i]
    print("Processing file {} named {}".format(i+1, filename))
    print("Voxel size: ", voxel_size)

    ### make mask of nuclei in ROI
    m = np.where(roi > 0, 1, 0)
    mask_3d, mask_3d_vol = measure_volume_fraction_from_masking(m, radius=10)
    masked_arr = roi*mask_3d

    ### measure concentration of nuclei in volume of the mask
    mask_3d_vol = mask_3d_vol*np.prod(voxel_size) 
    num_ids = len(np.unique(masked_arr))-1
    dens = num_ids/mask_3d_vol
    print("Concentration density is ", dens)

    ### measure volume fraction occupied by nuclei in the mask
    propsa = regionprops(masked_arr.astype('uint8'))
    volumes = [label.area for label in propsa]
    volumes = sum(volumes)*np.prod(voxel_size) 
    volume_fraction = volumes/mask_3d_vol
    print("Volume fraction is ", volume_fraction)

    ### save masks
    imwrite(os.path.join(packing_dir, filename[:-4]+'_mask3d.tif'), mask_3d, resolution=(1/voxel_size[2], 1/voxel_size[1]),
            metadata={'spacing': voxel_size[0],'axes': 'ZYX','unit': 'um'})

    ### save measurements in pandas dataframe
    myDict['hpf'].append(stage)
    myDict['density'].append(dens) #per squared micron
    myDict['volume_fraction'].append(volume_fraction)
    myDict['filename'].append(filename[:-4])
    temp_df = pd.DataFrame(myDict)
    temp_df.to_csv(name_csv)
   
### save pandas dataframe
df = pd.DataFrame(myDict)
df.to_csv(name_csv)

#### 3.2. Radial Distribution Function

The radial distribution function is computed for nuclei within the ROI.

In [ ]:
import pyclesperanto_prototype as cle
from skimage.morphology import closing, cube
from nuclear_packing import select_nuclei_in_roi, radial_distribution_function_3D

N = len(roi_images)
dr = 5
rMax = 200
edges = np.arange(0., rMax + 1.1 * dr, dr)
num_increments = len(edges) - 1

gs = np.empty((N, num_increments))
gs[:] = np.nan
ranges = np.zeros([N, num_increments])
stages = []

for i in range(N):
    # get filename and stage
    filename = os.path.basename(isoimages_files[i])[:-4]
    stage = str(filename[filename.find("hpf")-2:filename.find("hpf")])
    stages.append(stage)
    
    # get voxel size
    voxel_size = resolutions[i]
    
    # read labelled images
    #img = np.asarray(roi_images[i], dtype=int)
    roi = np.array(roi_images[i], dtype='int')
    m = np.where(roi > 0, 1, 0)
    #plt.imshow(np.max(m, axis=0))
    #plt.show()
    mgpu = cle.push_zyx(m)
    ext_mgpu = cle.extend_labels_with_maximum_radius(mgpu, radius = 20)
    mask_3d = closing(cle.pull(ext_mgpu), cube(100))
    #mask_3d, mask_3d_vol = measure_volume_fraction_from_masking(m, radius=20)
    
    masked_arr = roi*mask_3d
    mask_3d_vol = np.sum(mask_3d)*np.prod(voxel_size)

    labels = list(np.unique(masked_arr)[1:])
    pos_roi = select_nuclei_in_roi(roi,labels)
    #plt.imshow(np.max(pos_roi, axis=0))
    #plt.show()

   
    # get centroids
    roi_gpu = cle.push_zyx(roi)
    pointlist = cle.label_centroids_to_pointlist(roi_gpu)
    z = np.asarray(pointlist[2])
    y = np.asarray(pointlist[1])
    x = np.asarray(pointlist[0])

    indices = list(np.unique(pos_roi)[1:])
    num_ids = len(indices)
    dens = num_ids/mask_3d_vol
    print("Density", dens, num_ids, mask_3d_vol)

    (g_average, radii) = radial_distribution_function_3D(x, y, z, volMask=mask_3d_vol, rMax=rMax, dr=dr, interior_indices = indices)
    
    #print('prima', gs[i, :])
    gs[i, :rMax] = g_average
    #print('dopo', gs[i, :])
    ranges[i, :] = radii * voxel_size[0]


In [ ]:
# plot figure
fig, axes = plt.subplots(figsize=(6,6),  sharey=True)

averages = gs[0] #np.nanmedian(gs, axis=0)
#stds = np.nanstd(gs, axis=0)
axes.plot(ranges[0],averages*100, color='black', linewidth=3, label = 'average')
#axes.fill_between(ranges[0], averages*100 - stds*100, averages*100 + stds*100, color='black', alpha=0.1)
    
plt.xlabel('r')
plt.ylabel('g(r)')
plt.xlim( (0, 25) )
plt.ylim( (0, 1.05 * gs.max()) )
plt.show()

#### 3.3. Order parameter S

In [ ]:
from features import get_touch_matrix, angle_between

primary_files = sorted(glob(os.path.join(nematicDir, '*primary*.npy')))
secondary_files = sorted(glob(os.path.join(nematicDir, '*secondary*.npy')))

primary_arrays = list(map(np.load, primary_files))
secondary_arrays = list(map(np.load, secondary_files))
print(len(primary_arrays), primary_arrays[0].shape)
print(len(secondary_arrays), secondary_arrays[0].shape)

In [ ]:
order_parameter_s = {'nematics':[], 'order_parameter_s':[], 'order_parameter_s_other':[],  'median_angle':[], 'mean_angle':[], 'std_angle':[], 'stage':[]}
N = len(isoimages)
order_parameter_dict = {'stage':[], 'primary_mean_angle':[], 'primary_parameter_s':[], 'secondary_mean_angle':[], 'secondary_parameter_s':[] }

for j in range(len(roi_images)):
    filename = os.path.basename(isoimages_files[j])
   
    stage = int(filename[filename.find("hpf")-2:filename.find("hpf")])

    order_parameter_dict['stage'].append(stage)
    
    roi = roi_images[j]
    prim_arr = primary_arrays[j]
    sec_arr = secondary_arrays[j]
    prim_vects = np.abs(prim_arr[0]-prim_arr[1]) 
    sec_vects = np.abs(sec_arr[0]-sec_arr[1])
    
    roi_gpu = cle.push(roi)
    labels = list(np.unique(roi))
    touch_matrix, neighbors = get_touch_matrix(roi_gpu, dilation=1)

    d1, d2 = np.where(touch_matrix)

    prim_angles_dict = {}
    sec_angles_dict = {}
    for i in range(len(d1)):
        if d1[i] == 0:
            continue
        #print('--->>', d1[i], d2[i])
        a, b =  d1[i], d2[i]
        p = angle_between(prim_vects[a-1], prim_vects[b-1])
        s = angle_between(sec_vects[a-1], sec_vects[b-1])
        el = str(a)
        if el not in prim_angles_dict:
            prim_angles_dict[el] = [p]
            sec_angles_dict[el] = [s]
        else:
            prim_angles_dict[el].append(p)
            sec_angles_dict[el].append(s)

    p_means, s_means = [], []
    for k, values in prim_angles_dict.items():
        p_means.append(np.nanmean(values))
    for k, values in sec_angles_dict.items():
        s_means.append(np.nanmean(values))

    p_mean_val, s_mean_val = np.nanmean(p_means), np.nanmean(s_means)
    order_parameter_dict['primary_mean_angle'].append(p_mean_val)
    order_parameter_dict['secondary_mean_angle'].append(s_mean_val)
    order_parameter_dict['primary_parameter_s'].append((3*np.cos(p_mean_val)**2-1)/2)
    order_parameter_dict['secondary_parameter_s'].append((3*np.cos(s_mean_val)**2-1)/2)
    

In [ ]:
import matplotlib.patches as mpatches
import seaborn as sns


df_order_parameter = pd.DataFrame(order_parameter_dict)
df_order =  df_order_parameter[df_order_parameter.primary_parameter_s > 0.8]

sns.set(style="white")
fig, axes = plt.subplots( figsize=(6,6), sharey=True)
sns.swarmplot(data=df_order, x='stage', y='primary_parameter_s', s=8, dodge=True, alpha= 0.7)
sns.swarmplot(data=df_order, x='stage', y='secondary_parameter_s', s=8, dodge=True, alpha= 0.7)

sns.boxplot(showmeans=True,
            meanline=True,
            meanprops={'visible': False},
            medianprops={'color': 'k', 'ls': '--', 'lw': 1},
            whiskerprops={'visible': False},
            zorder=10,
            data=df_order, x='stage', y='primary_parameter_s',
            showfliers=False,
            showbox=False,
            showcaps=False,
            ax=axes)

sns.boxplot(showmeans=True,
            meanline=True,
            meanprops={'visible': False},
            medianprops={'color': 'k', 'ls': '--', 'lw': 1},
            whiskerprops={'visible': False},
            zorder=10,
            data=df_order, x='stage', y='secondary_parameter_s',
            showfliers=False,
            showbox=False,
            showcaps=False,
            ax=axes)

axes.tick_params(axis='both', labelsize=25)
axes.set_xlabel("Stages", fontsize=25)
axes.set_ylabel( 'Order parameter S', fontsize=25)
#axes.set_xticklabels(categories)
a_patch = mpatches.Patch(color='darkcyan', label='Primary')
b_patch = mpatches.Patch(color='orange', label='Secondary')
plt.legend(handles=[a_patch, b_patch])
plt.yticks([ 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]) #np.arange(0.9, 1, 0.5))
#plt.savefig('path/to/plots/'+date+'_local_order_parameter_both_neighbors.svg', bbox_inches = 'tight')
plt.show() 